# Text embeddings: a first look

Four short clinical texts, each turned into **one vector** (an embedding)
by a language model, and two things done with those vectors: look at them
and compare them. Then the same steps with other models, to see why some
models give useful vectors and others do not.
Classifying the texts comes next, in `../classification/title-classification.ipynb`.

The texts are sentences from four cases of the MultiCaRe sample in
`../../data/multicare/data/csv_sample/` (cases `PMC5137649_01`, `PMC11722600_01`, `PMC6949661_01`,
`PMC6134951_01`).
Texts 1 and 2 describe a woman with abdominal pain; text 3 describes a man
found confused, possibly after a seizure. Text 4 describes a medication and
its effects.

## 1. Preparation

Libraries and the four texts. The model is chosen in the next section.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModel

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
pd.set_option("display.max_columns", 10)          # show the first and last dimensions only

TEXTS = {
    "Text 1": "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant "
              "abdominal pain associated with nausea and constipation.",
    "Text 2": "A 41-year-old female with a history of cesarean section presented to the emergency "
              "department with continuous abdominal pain for 12 h, localized in the right iliac fossa (RIF).",
    "Text 3": "A 23-year-old male was brought to the emergency of Hamad General Hospital in Doha, after "
              "being found on the floor dizzy and confused, with urinary incontinence although no one "
              "witnessed convulsions.",
    "Text 4": "However, 10 days later, she developed a non-pruritic morbilliform rash spreading from trunk "
              "to the entire body, including palms, soles, and face with mild lip and eye swelling."
}

## 2. From text to embedding, one text at a time

**Choose a model.** The next cell lists four models: one line is active,
the others are comments. To try another model, comment the active line,
uncomment the one you want, and run the notebook again from that cell
(*Run → Run Selected Cell and All Below*).

|   | model | size | trained to… | pooling |
|---|---|---|---|---|
| **A** | `BAAI/bge-small-en-v1.5` | 33M parameters, 384 dimensions | give texts with similar meanings similar vectors (contrastive learning on millions of text pairs) | `cls` |
| **B** | `distilbert-base-uncased` | 66M, 768 | predict masked words in general English (Wikipedia, books), imitating BERT | `cls` |
| **C** | `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` | 110M, 768 | predict masked words in PubMed abstracts and articles, and whether one sentence follows another | `cls` |
| **D** | `NeuML/pubmedbert-base-embeddings` | 110M, 768 | **C**, then trained like **A**: similar vectors for a PubMed title and its abstract | `mean` |

The model returns one vector per token; **pooling** is how a single vector
for the whole text is taken from them:

* `cls` — keep the vector of the `[CLS]` token (position 0);
* `mean` — average the vectors of all tokens.

A model trained to produce embeddings is trained with one of the two, and
its model card says which: `cls` for A, `mean` for D. B and C were never
trained to produce a vector for a text's meaning (C's `[CLS]` was trained
only to tell whether two sentences follow each other), so neither choice is
"theirs"; `cls` is used here, and you can try `mean`.

The basic cycle, three steps per text:

1. **tokenize** — the text becomes token ids, with `[CLS]` added at the
   start and `[SEP]` at the end;
2. **encode** — the model returns one contextual vector per token;
3. **pool** — one vector for the whole text: the embedding.

In [2]:
# Choose ONE model (see the table above): uncomment its line, comment the others.
MODEL_NAME, POOLING = "BAAI/bge-small-en-v1.5", "cls"                                          # A
# MODEL_NAME, POOLING = "distilbert-base-uncased", "cls"                                       # B
# MODEL_NAME, POOLING = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext", "cls"  # C
# MODEL_NAME, POOLING = "NeuML/pubmedbert-base-embeddings", "mean"                             # D

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()   # eval(): inference only, no training

embeddings = {}

for name, text in TEXTS.items():
    inputs = tokenizer(text, return_tensors="pt")          # 1. text → token ids
    with torch.no_grad():                                  #    (no gradients: we are not training)
        outputs = model(**inputs)                          # 2. token ids → one vector per token
    vectors = outputs.last_hidden_state[0]                 #    text 0: (tokens × dimensions)
    if POOLING == "cls":
        embedding = vectors[0]                             # 3. position 0 = the [CLS] vector
    else:
        embedding = vectors.mean(dim=0)                    # 3. the average of all token vectors
    embeddings[name] = embedding

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    print(f"{name}: {len(tokens)} tokens {tokens[:4]} ... {tokens[-2:]}")
    print(f"        all token vectors: {tuple(vectors.shape)}  →  embedding ({POOLING}): {tuple(embedding.shape)}")

Text 1: 33 tokens ['[CLS]', 'a', '44', '-'] ... ['.', '[SEP]']
        all token vectors: (33, 384)  →  embedding (cls): (384,)
Text 2: 42 tokens ['[CLS]', 'a', '41', '-'] ... ['.', '[SEP]']
        all token vectors: (42, 384)  →  embedding (cls): (384,)
Text 3: 49 tokens ['[CLS]', 'a', '23', '-'] ... ['.', '[SEP]']
        all token vectors: (49, 384)  →  embedding (cls): (384,)
Text 4: 43 tokens ['[CLS]', 'however', ',', '10'] ... ['.', '[SEP]']
        all token vectors: (43, 384)  →  embedding (cls): (384,)


## 3. Looking at the embeddings

The simplest view is a table: **one row per text, one column per
dimension**. Only the first and last dimensions are shown (`...` hides the
rest). A single number means nothing by itself; the meaning is in the
whole row.

In [3]:
table = pd.DataFrame(torch.stack(list(embeddings.values())).numpy(), index=embeddings.keys())
print("shape:", table.shape, "(texts × dimensions)")
table.round(3)

shape: (4, 384) (texts × dimensions)


,0,1,2,3,4,...,379,380,381,382,383
Text 1,-0.131,-0.179,0.051,0.836,-0.018,...,-0.092,-0.141,-0.546,0.248,-0.185
Text 2,0.041,-0.415,-0.067,0.476,0.361,...,0.037,-0.255,-0.296,0.492,-0.530
Text 3,0.157,0.298,-0.094,0.291,0.359,...,-0.172,-0.136,0.266,-0.372,-0.492
Text 4,-0.083,0.315,0.161,0.250,-0.428,...,0.314,-0.602,-0.956,-0.334,0.645


## 4. Comparing texts

The **cosine similarity** of two vectors $a$ and $b$ is
$\dfrac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$: 1 when they point in the same
direction, 0 when they are perpendicular.

The last line, the **spread** (highest minus lowest similarity), sums up
how much the model tells the pairs apart.

In [4]:
names = list(embeddings)
similarities = []

for i, a in enumerate(names):
    for b in names[i + 1:]:
        similarity = F.cosine_similarity(embeddings[a], embeddings[b], dim=0).item()
        similarities.append(similarity)
        print(f"{a} × {b}: {similarity:.3f}")

print(f"\n{MODEL_NAME} ({POOLING}) — spread: {max(similarities) - min(similarities):.3f}")

Text 1 × Text 2: 0.791
Text 1 × Text 3: 0.632
Text 1 × Text 4: 0.575
Text 2 × Text 3: 0.710
Text 2 × Text 4: 0.549
Text 3 × Text 4: 0.521

BAAI/bge-small-en-v1.5 (cls) — spread: 0.270


## What to notice (model A)

* **Texts of any length, vectors of one size.** The four texts have 33,
  42, 49 and 43 tokens, so the model returns 33, 42, 49 and 43 vectors; pooling
  gives each text the same 384 numbers.
* **The numbers are not readable one by one.** No column of the table
  stands for "abdomen" or "seizure"; only comparisons between whole rows
  carry meaning.
* **Texts 1 and 2 are the closest pair** (0.791), and text 1 is farthest
  from text 3 (0.632). Text 2 sits closer to text 3 (0.710) than text 1
  does: both mention *the emergency* and say how the patient arrived,
  while text 1 has only symptoms.
* **Text 4 is the farthest from all the others** (0.521 to 0.575): it is
  about a skin reaction, and it is not the opening of a case — no
  "A NN-year-old ... presented".
* **Similarities do not go down to 0.** Unrelated clinical sentences still
  share a lot (the genre, "A NN-year-old ..."), so this model's scores live
  in a narrow upper range; what matters is the *order* of the pairs, not the
  value of one.

## 5. Comparing models

Run the notebook once per model and note the results in the table below
(double-click this cell to edit it). Row A is filled in from the run above.
For C, run it twice: with `cls`, then with `mean` (change `"cls"` to
`"mean"` in its line).

|   | model | pooling | 1×2 | 1×3 | 1×4 | 2×3 | 2×4 | 3×4 | spread |
|---|---|---|---|---|---|---|---|---|---|
| A | bge-small-en-v1.5 | `cls` | 0.791 | 0.632 | 0.575 | 0.710 | 0.549 | 0.521 | 0.270 |
| B | distilbert-base-uncased | `cls` | | | | | | | |
| C | BiomedBERT | `cls` | | | | | | | |
| C | BiomedBERT | `mean` | | | | | | | |
| D | pubmedbert-base-embeddings | `mean` | | | | | | | |

Questions:

1. For which models are texts 1 and 2 still the closest pair, and text 4
   the farthest from the others? For which ones do all six similarities
   look alike?
2. B and C were both trained to predict masked words, C on medical text.
   Does knowing medical vocabulary make C better at comparing these texts?
3. C and D have the same architecture, the same vocabulary and start from
   the same weights; only D's extra training differs. What does that say
   about where the ability to compare texts comes from?
4. Does `mean` pooling change the picture for C?
5. Is a similarity of 0.8 "high"? Can you compare a value from one model
   with a value from another?

Next: `../classification/title-classification.ipynb` trains a logistic
regression on the `[CLS]` vectors of model A (1,600 labeled medical titles)
and uses it to route these cases.